In [ ]:
%load_ext autoreload
%autoreload 2

print("Autoreload extension loaded. All modules will be reloaded automatically before executing code.")

In [1]:
import logging
import torch
from torchinfo import summary
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import LitLogger

from workspace.config import Config
from workspace.dataset import MNISTDataset
from workspace.model import MNISTClassifier

torch.set_float32_matmul_precision("medium")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = LitLogger(
    root_dir="data/logs",
    name="deepdive",
)

config = Config()
dataset = MNISTDataset(config)

checkpointer = ModelCheckpoint(
    dirpath=config.training.checkpoint.rootdir,
    filename="mnist-{epoch:02d}",
    save_last=True,
    save_top_k=1,
)

trainer = pl.Trainer(
    accelerator="cuda",
    devices=[0],
    max_epochs=config.training.max_epochs,
    callbacks=[checkpointer],
    deterministic=True,
    enable_progress_bar=True,
    enable_model_summary=False,
    logger=logger,
    precision="16-mixed",
)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [2]:
model = MNISTClassifier(
    layer_depth=config.model.layer_depth,
    kernel_depth=config.model.kernel_depth,
    learning_rate=config.training.optimizer.learning_rate,
    weight_decay=config.training.optimizer.weight_decay,
)

print("\nLoaded", config, "\n")
summary(model, input_size=(config.training.batch_size, 1, 28, 28), verbose=1)

trainer.fit(model, datamodule=dataset)

validation_metrics = trainer.validate(model, datamodule=dataset, ckpt_path="last", verbose=False)
print("Validation metrics:", validation_metrics)



Loaded Config {
    "data": {
        "rootdir": "/workspaces/deepdive/data/datasets",
        "dataset": "mnist"
    },
    "model": {
        "layer_depth": 3,
        "kernel_depth": 64
    },
    "training": {
        "max_epochs": 10,
        "batch_size": 64,
        "optimizer": {
            "learning_rate": 0.0005,
            "weight_decay": 0.0,
            "scheduler": {
                "step_size": 5,
                "gamma": 0.1
            }
        },
        "checkpoint": {
            "rootdir": "/workspaces/deepdive/data/checkpoints",
            "interval": 1,
            "restore": "last"
        }
    },
    "logging": {
        "level": "INFO",
        "log_every_n_steps": 10
    },
    "runtime": {
        "seed": 42,
        "pin_memory": true,
        "num_workers": 0,
        "device": "cuda",
        "version": "2.13.0+cu130"
    }
} 

Layer (type:depth-idx)                   Output Shape              Param #
MNISTClassifier                          [64, 10

AttributeError: 'DataConfig' object has no attribute 'transform'